  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataiku/kiji-inspector/blob/main/demo/quickstart_colab.ipynb)

In [1]:
!pip install -U -q kiji-inspector transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 142.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 48.5 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor


LAYER_INDEX = 8
MODEL_ID = "google/gemma-4-E4B-it"
PROMPT = "My dishwasher is smelly, what is the first element I should review?"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

messages = [{"role": "user", "content": [{"type": "text", "text": PROMPT}]}]
prompt = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = processor(text=prompt, return_tensors="pt")
inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

captured = {}

def hook(_module, _inputs, output):
    hidden = output[0] if isinstance(output, tuple) else output
    captured["tensor"] = hidden.detach().cpu()

layer = model.model.language_model.layers[LAYER_INDEX]
handle = layer.register_forward_hook(hook)
try:
    with torch.inference_mode():
        model(**inputs)
finally:
    handle.remove()

hidden_state = captured["tensor"]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

In [3]:
inputs['input_ids'].shape

torch.Size([1, 23])

In [4]:
hidden_state.shape

torch.Size([1, 23, 2560])

In [5]:
from kiji_inspector import SAE

sae, feature_descriptions = SAE.from_pretrained(
    base_model="google/gemma-4-E4B-it",
    layer=8,
)

# Extract the activation for the first sequence, last token
last_token_act = hidden_state[0, -1, :]

# Describe the top features activating on this token using the descriptions dictionary
sae.describe(last_token_act, feature_descriptions)


layer_8/sae_checkpoints/sae_final.pt:   0%|          | 0.00/168M [00:00<?, ?B/s]

feature_descriptions.json: 0.00B [00:00, ?B/s]

[(2341, 'unknown', 7.579923629760742),
 (14641,
  {'label': 'Dishwasher Gasket Issues',
   'description': 'This feature detects descriptions of dishwashers leaking from the door or bottom, often mentioning a worn, torn, or cracked door gasket.',
   'confidence': 'high',
   'mean_activation': 7.65625,
   'max_activation': 8.625,
   'frac_nonzero': 1.0,
   'top_examples': ["My 18-year-old dishwasher is leaking from the door and the tub looks rusted; I think it's a worn gasket.",
    'My 18-year-old dishwasher is leaking from the door; I think the door gasket is worn out and needs replacing.',
    "My 18-year-old dishwasher is leaking from the bottom and the tub looks rusted; I think it's a worn door gasket.",
    "My 18-year-old dishwasher is leaking from the door and the tub looks corroded; I'm wondering if the door gasket is torn.",
    'My 14-year-old dishwasher is leaking from the door and the tub looks corroded; I think the door gasket is torn and needs replacing.',
    "My 14-year-

In [6]:
results = sae.describe(last_token_act, feature_descriptions)

for feature_id, desc, activation in results:
    print(f"Feature ID: {feature_id} | Activation: {activation:.2f}")
    if isinstance(desc, dict):
        print(f"  Label: {desc.get('label', 'N/A')}")
        print(f"  Description: {desc.get('description', 'N/A')}")
        print(f"  Confidence: {desc.get('confidence', 'N/A')}")
    else:
        print(f"  Label: {desc}")
    print("-" * 50)

Feature ID: 2341 | Activation: 7.58
  Label: unknown
--------------------------------------------------
Feature ID: 14641 | Activation: 7.19
  Label: Dishwasher Gasket Issues
  Description: This feature detects descriptions of dishwashers leaking from the door or bottom, often mentioning a worn, torn, or cracked door gasket.
  Confidence: high
--------------------------------------------------
Feature ID: 13122 | Activation: 7.12
  Label: unknown
--------------------------------------------------
Feature ID: 131 | Activation: 6.99
  Label: unknown
--------------------------------------------------
Feature ID: 5101 | Activation: 6.83
  Label: unknown
--------------------------------------------------


---

In [7]:
del model

In [8]:
!pip install -q "torch>=2.11.0" "torchvision>=0.26.0" "transformers>=5.0.0" "accelerate>=1.13.0" "vllm==0.20.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 144.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/3

In [9]:
!wget -q https://raw.githubusercontent.com/dataiku/kiji-inspector/279e4c6b3d26b2db0b87a053a192cc90f893b520/patches/0.20.1/apply_patch.sh
!wget -q https://raw.githubusercontent.com/dataiku/kiji-inspector/279e4c6b3d26b2db0b87a053a192cc90f893b520/patches/0.20.1/vllm-0.20.1-gemma4-hidden-states.patch
!bash apply_patch.sh

Detected vllm install:
  python : python3
  path   : /usr/local/lib/python3.12/dist-packages/vllm
  version: 0.20.1
  patch  : /content/vllm-0.20.1-gemma4-hidden-states.patch

Dry-run check (no files modified yet)...
checking file config/model.py
checking file engine/arg_utils.py
checking file entrypoints/llm.py
checking file entrypoints/openai/chat_completion/protocol.py
checking file entrypoints/openai/chat_completion/serving.py
checking file entrypoints/openai/cli_args.py
checking file entrypoints/openai/completion/protocol.py
checking file entrypoints/openai/completion/serving.py
checking file entrypoints/openai/engine/serving.py
checking file entrypoints/openai/generate/api_router.py
checking file model_executor/models/gemma4.py
checking file outputs.py
checking file v1/core/sched/scheduler.py
checking file v1/engine/__init__.py
checking file v1/engine/output_processor.py
checking file v1/outputs.py
checking file v1/worker/gpu_input_batch.py
checking file v1/worker/gpu_model_runne

In [1]:
import os

import torch

os.environ.setdefault("VLLM_ALLOW_INSECURE_SERIALIZATION", "1")
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import LLM, SamplingParams

MODEL_ID = "google/gemma-4-E4B-it"
PROMPT = "The capital of France is"
LAYER_INDEX = 8

llm = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    dtype="float16",
    tensor_parallel_size=1,
    extract_activation_layers=[LAYER_INDEX],
)

outputs = llm.generate(
    [PROMPT],
    SamplingParams(max_tokens=1, temperature=0.0),
    use_tqdm=False,
)

completion = outputs[0].outputs[0]
hidden_state = completion.activations[LAYER_INDEX].to(device="cpu", dtype=torch.float32)

print(f"Hidden state shape: {tuple(hidden_state.shape)}")


INFO 05-20 06:35:02 [utils.py:233] non-default args: {'extract_activation_layers': [8], 'trust_remote_code': True, 'dtype': 'float16', 'disable_log_stats': True, 'model': 'google/gemma-4-E4B-it'}
INFO 05-20 06:35:22 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-20 06:35:22 [nixl_utils.py:34] NIXL is not available
WARNING 05-20 06:35:22 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-20 06:35:22 [model.py:561] Resolved architecture: Gemma4ForConditionalGeneration
WARNING 05-20 06:35:22 [model.py:2051] Casting torch.bfloat16 to torch.float16.
INFO 05-20 06:35:22 [model.py:1713] Using max model len 131072
INFO 05-20 06:35:22 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-20 06:35:22 [config.py:101] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
I